# DistilBERT Transformer Baseline — Multi-Label Toxicity

Exploratory notebook comparing a transformer fine-tune against the BiLSTM pipelines already in this project.

**Why**: the BiLSTM models get strong ROC-AUC (~0.96) but weak PR-AUC on the rare classes (`identity_attack` ~0.25, `threat` ~0.30 — see `models/metrics.json`). A transformer's pretrained language understanding should generalize better on these sparse classes with the same training data.

**Model**: `distilbert-base-uncased` (66M params) — small enough for CPU inference in production (matches the deployed Hugging Face Spaces CPU tier), while still being a real transformer with self-attention we can use for explainability (see the attention-rollout section at the end).

**Labels** (same as the BiLSTM models): `toxicity, obscene, sexual_explicit, identity_attack, insult, threat` — 6 independent sigmoid outputs (multi-label, not multi-class).

Run in the `toxguard-transformer` conda env (GPU training; torch + transformers).

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 0. Config

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128          # DistilBERT default context is 512, but comments are short; 128 keeps training fast
BATCH_SIZE = 32
EPOCHS = 3             # transformers fine-tune fast; more epochs risks overfitting on this much data
LEARNING_RATE = 2e-5   # standard fine-tuning LR for BERT-family models
TEXT_COL = "comment_text"
TARGET_COLS = ['toxicity', 'obscene', 'sexual_explicit', 'identity_attack', 'insult', 'threat']
NUM_CLASSES = len(TARGET_COLS)

# For notebook exploration we optionally subsample so iteration is fast; the
# production script (scripts/train_transformer.py) trains on the FULL split.
SUBSAMPLE_TRAIN = 150_000   # set to None to use the full training set

## 1. Load data

Same splits used by the BiLSTM models — no re-splitting, so all three models are evaluated on the exact same held-out test set.

In [ ]:
train_df = pd.read_csv("../data/input/train_split.csv").fillna({TEXT_COL: "missing_text"})
val_df = pd.read_csv("../data/input/val_split.csv").fillna({TEXT_COL: "missing_text"})
test_df = pd.read_csv("../data/input/test_split.csv").fillna({TEXT_COL: "missing_text"})

if SUBSAMPLE_TRAIN is not None and len(train_df) > SUBSAMPLE_TRAIN:
    train_df = train_df.sample(n=SUBSAMPLE_TRAIN, random_state=SEED).reset_index(drop=True)

print(f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
train_df.head(3)

## 2. Tokenizer & Dataset

DistilBERT uses WordPiece tokenization (subword units). Unlike the BiLSTM's `TextVectorization` (which builds its own 20k-word vocabulary from scratch), the tokenizer here is **pretrained** — it already knows ~30k subword tokens from DistilBERT's original pretraining corpus, which is a big part of why transformers generalize better on rare words/classes.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- Performance notes ---
# 1) Tokenizing one string at a time inside Dataset.__getitem__ serializes CPU
#    work in front of every GPU step. We tokenize the WHOLE split ONCE upfront
#    (to token-ID lists, no padding yet) so __getitem__ is pure indexing.
# 2) We do NOT pad every sample to MAX_LEN. Most comments are far shorter than
#    128 tokens, so fixed max-length padding wastes a large fraction of every
#    forward/backward pass on padding tokens. Instead, padding happens per-batch
#    in collate_fn, padding only to that batch's own longest sequence.
def pretokenize(texts, tokenizer, max_len):
    texts = [str(t) for t in texts]
    enc = tokenizer(texts, truncation=True, max_length=max_len)
    return enc["input_ids"]

class ToxicityDataset(Dataset):
    """Wraps PRE-TOKENIZED (unpadded) id lists — __getitem__ is just indexing."""
    def __init__(self, input_ids, labels):
        self.input_ids = input_ids
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "labels": self.labels[idx]}

def make_collate_fn(tokenizer):
    """Dynamically pads each batch to its own longest sequence."""
    def collate(batch):
        ids = [item["input_ids"] for item in batch]
        labels = torch.stack([item["labels"] for item in batch])
        padded = tokenizer.pad({"input_ids": ids}, padding=True, return_tensors="pt")
        return {"input_ids": padded["input_ids"], "attention_mask": padded["attention_mask"], "labels": labels}
    return collate

collate_fn = make_collate_fn(tokenizer)

print("Tokenizing train/val/test splits upfront (one-time cost, then training is GPU-bound)...")
t0 = time.time()
train_ids = pretokenize(train_df[TEXT_COL].tolist(), tokenizer, MAX_LEN)
val_ids = pretokenize(val_df[TEXT_COL].tolist(), tokenizer, MAX_LEN)
test_ids = pretokenize(test_df[TEXT_COL].tolist(), tokenizer, MAX_LEN)
print(f"  done in {time.time() - t0:.0f}s")

train_ds = ToxicityDataset(train_ids, train_df[TARGET_COLS].values)
val_ds = ToxicityDataset(val_ids, val_df[TARGET_COLS].values)
test_ds = ToxicityDataset(test_ids, test_df[TARGET_COLS].values)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collate_fn)

# Quick sanity check of what tokenization looks like
sample = tokenizer("You are such an idiot!!", return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(sample["input_ids"][0]))

## 3. Model architecture

`DistilBERT backbone -> take the [CLS] token's final hidden state -> dropout -> linear(768 -> 6) -> sigmoid (via BCEWithLogitsLoss)`.

This is the standard "add a classification head on top of a pretrained encoder" pattern. We fine-tune the WHOLE backbone (not just the head) — small learning rate (2e-5) keeps this stable.

In [ ]:
class DistilBertMultiLabel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size  # 768 for distilbert-base
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, output_attentions=False):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
        )
        cls_hidden = out.last_hidden_state[:, 0, :]   # [CLS] token representation
        logits = self.classifier(self.dropout(cls_hidden))
        if output_attentions:
            return logits, out.attentions   # tuple of 6 layers, each [batch, heads, seq, seq]
        return logits

model = DistilBertMultiLabel(MODEL_NAME, NUM_CLASSES).to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Class-imbalance handling

Same idea as the BiLSTM scripts (bias-init + no plain class-weighting), adapted to PyTorch: `BCEWithLogitsLoss(pos_weight=...)` upweights the rare positive classes directly in the loss, mirroring what focal loss + bias-init achieved for the BiLSTMs. `pos_weight[c] = neg_count[c] / pos_count[c]`.

In [ ]:
pos_counts = train_df[TARGET_COLS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float).to(DEVICE)
print(dict(zip(TARGET_COLS, pos_weight.cpu().numpy().round(2))))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## 5. Training loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

def run_epoch(loader, train=True, log_every=200):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    n_batches = len(loader)
    t0 = time.time()
    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attn = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(input_ids, attn)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

        total_loss += loss.item() * len(labels)
        all_logits.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_labels.append(labels.detach().cpu().numpy())

        if train and (step + 1) % log_every == 0:
            elapsed = time.time() - t0
            rate = (step + 1) / elapsed
            eta = (n_batches - step - 1) / rate
            print(f"    batch {step+1}/{n_batches}  loss={loss.item():.4f}  {rate:.1f} batch/s  ETA this epoch: {eta/60:.1f} min")

    avg_loss = total_loss / len(loader.dataset)
    probs = np.concatenate(all_logits)
    y = np.concatenate(all_labels)
    aucs = [roc_auc_score(y[:, j], probs[:, j]) if len(np.unique(y[:, j])) > 1 else float("nan") for j in range(NUM_CLASSES)]
    return avg_loss, np.nanmean(aucs), probs, y

history = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_auc, _, _ = run_epoch(train_loader, train=True)
    val_loss, val_auc, val_probs, val_y = run_epoch(val_loader, train=False)
    dt = time.time() - t0
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_auc": val_auc})
    print(f"epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_mean_auc={val_auc:.4f}  ({dt:.0f}s)")

## 6. Per-class optimal thresholds (same convention as the BiLSTM artifacts)

In [ ]:
optimal_thresholds = {}
for j, c in enumerate(TARGET_COLS):
    precision, recall, thresh = precision_recall_curve(val_y[:, j], val_probs[:, j])
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(precision), where=(precision + recall) != 0)
    best_idx = np.argmax(f1[:-1])
    optimal_thresholds[c] = float(thresh[best_idx])
    print(f"{c:<18} optimal_threshold={thresh[best_idx]:.4f}  f1={f1[best_idx]:.4f}")

## 7. Held-out test evaluation (same test set as the BiLSTM comparison in models/metrics.json)

In [ ]:
_, test_auc, test_probs, test_y = run_epoch(test_loader, train=False)
for j, c in enumerate(TARGET_COLS):
    auc = roc_auc_score(test_y[:, j], test_probs[:, j])
    ap = average_precision_score(test_y[:, j], test_probs[:, j])
    print(f"{c:<18} ROC-AUC={auc:.4f}  PR-AUC={ap:.4f}")

## 8. Explainability preview — attention rollout

Raw single-layer attention is noisy. **Attention rollout** (Abnar & Zuidema, 2020) recursively multiplies the attention matrices across all 6 layers (averaging heads, adding the identity matrix to account for residual connections), producing one matrix that represents cumulative attention flow from input tokens to the `[CLS]` token that the classifier actually reads. The `[CLS]` row of that rolled-out matrix is our per-token importance score. Subword tokens are summed back into whole words for a readable highlight. The production version of this lives in `app/explain.py` and is used by the API/UI.

In [ ]:
def attention_rollout(attentions):
    """attentions: tuple of [1, heads, seq, seq] tensors, one per layer."""
    seq_len = attentions[0].shape[-1]
    rollout = torch.eye(seq_len).to(attentions[0].device)
    for layer_attn in attentions:
        avg_heads = layer_attn.mean(dim=1)[0]              # average over heads -> [seq, seq]
        avg_heads = avg_heads + torch.eye(seq_len).to(avg_heads.device)  # residual connection
        avg_heads = avg_heads / avg_heads.sum(dim=-1, keepdim=True)      # re-normalize rows
        rollout = avg_heads @ rollout
    return rollout[0]  # CLS token's row = importance of every token to the final decision

sample_text = "You are such a worthless idiot and everyone hates you"
enc = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(DEVICE)
model.eval()
with torch.no_grad():
    logits, attentions = model(enc["input_ids"], enc["attention_mask"], output_attentions=True)

cls_scores = attention_rollout(attentions).cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
for tok, score in zip(tokens, cls_scores):
    if tok not in (tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token):
        print(f"{tok:<15} {score:.4f}")

probs = torch.sigmoid(logits)[0].detach().cpu().numpy()
print("\nPredicted probabilities:", dict(zip(TARGET_COLS, probs.round(3))))

## 9. Save model + artifacts

This notebook trained on a subsample for fast iteration. The **production run** (full training data, saved to `models/transformer_model/`) happens via `scripts/train_transformer.py` — the clean, script-ified version of everything above, meant to be run non-interactively and to be the actual artifact the API loads.

Uncomment below only if you want THIS notebook's (subsampled) run to become the deployed model.

In [ ]:
# import os
# os.makedirs("../models/transformer_model", exist_ok=True)
# model.backbone.save_pretrained("../models/transformer_model")
# tokenizer.save_pretrained("../models/transformer_model")
# torch.save(model.classifier.state_dict(), "../models/transformer_model/classifier_head.pt")
# with open("../models/transformer_inference_artifacts.json", "w") as f:
#     json.dump({"optimal_thresholds": optimal_thresholds, "target_cols": TARGET_COLS}, f, indent=4)
# print("Saved (notebook run).")